# Implement Muon from Scratch - SOLUTION

**Difficulty**: 🔴 Hard

**Companies**: Meta, Google

---

### Problem Statement

Muon ("Momentum Orthogonalized by Newton–Schulz", Jordan et al., 2024) is the optimizer behind recent LLM training speedrun records:

- **Matrix parameters (`ndim >= 2`)** — momentum, then orthogonalize the update via Newton–Schulz, then apply it.
- **1D parameters** (biases, norm weights) — fall back to AdamW.

In real training Muon is meant for the *hidden* weight matrices only: embeddings, the output head/classifier and all 1D parameters go to AdamW (with its own, smaller learning rate, via param groups). Dispatching on `ndim` as below is a simplification of that rule.

### Tasks

1. `newton_schulz(X, steps)` — approximate matrix orthogonalization (the helper Muon needs).
2. `MyMuon` — momentum followed by orthogonalization for matrix parameters; an AdamW fallback for 1D parameters.

### References

- Muon blog: https://kellerjordan.github.io/posts/muon/
- Reference implementation: https://github.com/KellerJordan/Muon (PyTorch ≥ 2.9 also ships it as `torch.optim.Muon`, for 2D parameters only)
- Muon is Scalable for LLM Training (Moonshot AI, 2025): https://arxiv.org/abs/2502.16982

In [ ]:
import math
import torch
import torch.nn as nn
from torch.optim import Optimizer


## Helper: Newton–Schulz Orthogonalization

Muon replaces the update matrix `G = U S Vᵀ` by (approximately) its orthogonal polar factor `U Vᵀ` — same shape, same singular vectors, but every singular value set to ~1. SVD would do it exactly but is too slow inside a training loop; Muon uses a quintic **Newton–Schulz iteration**, which needs only matmuls:

```
A = X Xᵀ
X ← a·X + b·A·X + c·A²·X        (a, b, c) = (3.4445, −4.7750, 2.0315)
```

Each step applies the odd polynomial `p(σ) = aσ + bσ³ + cσ⁵` to every singular value of `X` and leaves the singular vectors alone.

- The textbook coefficients `(15/8, −5/4, 3/8)` satisfy `p(1) = 1, p′(1) = 0`, so they converge to exactly 1 — but small singular values only grow by 15/8 per step.
- Muon's coefficients (Keller Jordan) are tuned for a large slope at 0 (≈3.4 per step), so 5 steps are enough for every singular value above about 0.003·‖X‖_F. The price: they do not converge exactly — after 5 steps those singular values land roughly in **[0.7, 1.2]** (a singular value that starts near 0 stays small, whatever the coefficients). That empirically does not hurt training. Use Muon's coefficients.

The iteration only behaves when the spectral norm of the input is at most 1, so normalize first — the Frobenius norm is a cheap upper bound on the spectral norm (add a small `eps` so a zero matrix stays zero).

**Tip:** for an `(m, n)` matrix, `X @ X.T` is `(m, m)` and `X.T @ X` is `(n, n)` — transpose a tall matrix first so the Gram matrix is the *smaller* one, and transpose back at the end.

In [ ]:
def newton_schulz(X: torch.Tensor, steps: int = 5, eps: float = 1e-7) -> torch.Tensor:
    """
    Approximately orthogonalize a matrix with the quintic Newton–Schulz iteration.

    Given X = U S Vᵀ (m × n), return ≈ U S' Vᵀ of the same shape, where every
    singular value in S' has been pushed to roughly 1 (with Muon's
    coefficients they land in about [0.7, 1.2]) — an approximation of the
    orthogonal polar factor U Vᵀ.

    Uses Muon's coefficients (a, b, c) = (3.4445, -4.7750, 2.0315).

    Args:
        X:     input matrix, shape (m, n)
        steps: number of Newton–Schulz iterations (default 5)
        eps:   added to the norm so a zero matrix does not divide by zero

    Returns:
        Orthogonalized matrix, same shape as X
    """
    a, b, c = 3.4445, -4.7750, 2.0315

    # Work in the wide orientation so the Gram matrix A = X @ X.T is the small one
    transposed = X.shape[0] > X.shape[1]
    if transposed:
        X = X.T

    # Normalize so the spectral norm is <= 1 (Frobenius norm >= spectral norm)
    X = X / (X.norm() + eps)

    for _ in range(steps):
        A = X @ X.T                    # (k, k) with k = min(m, n)
        B = b * A + c * (A @ A)
        X = a * X + B @ X              # applies p(σ) = aσ + bσ³ + cσ⁵ to every singular value

    return X.T if transposed else X

## Muon

Muon ("Momentum Orthogonalized by Newton–Schulz") is a modern optimizer for the hidden weight matrices of large transformers.

**Matrix parameters (`ndim >= 2`):**

1. Momentum: `buf ← μ·buf + g` — with Nesterov look-ahead, the update is `g + μ·buf`, otherwise just `buf`. (Keller Jordan's code writes it as an EMA, `buf ← μ·buf + (1−μ)·g`; that only rescales the update, and the normalization inside Newton–Schulz removes any scale.)
2. Orthogonalize the update with your `newton_schulz`. Conv filters `(out, in, kh, kw)` are treated as an `(out, in·kh·kw)` matrix.
3. Rescale by `√max(1, rows / cols)` (Keller Jordan's reference, and `torch.optim.Muon`'s default). An orthogonalized update has all singular values ≈ 1; for a tall matrix (fan-out > fan-in) that would make its effect on the layer's output RMS too small, so tall matrices are scaled up. Other published conventions: `0.2·√max(rows, cols)` (Moonshot — matches AdamW's update RMS so AdamW's lr and weight decay can be reused) and `√(rows / cols)` (Bernstein).
4. Apply the same decoupled weight decay as AdamW, then `θ ← θ − lr·scale·update`.

Muon keeps **one** state buffer per matrix (Adam keeps two) — don't allocate Adam buffers for matrices.

**1D parameters** (biases, LayerNorm gains): orthogonalization is meaningless for a vector, so fall back to an **AdamW-style** update with its own betas and eps.

Typical hyperparameters (Keller Jordan's defaults): `lr=0.02, momentum=0.95, nesterov=True, ns_steps=5` (his default `weight_decay` is 0).

In [ ]:
class MyMuon(Optimizer):
    """
    Muon optimizer — Momentum with Newton–Schulz Orthogonalization.

    Args:
        params:       iterable of parameters to optimize
        lr:           learning rate (default 2e-2 — higher than Adam)
        momentum:     momentum coefficient μ (default 0.95)
        nesterov:     use Nesterov-style look-ahead momentum (default True)
        ns_steps:     Newton–Schulz iterations (default 5)
        weight_decay: decoupled weight decay (default 1e-2)
        adamw_betas:  betas for the AdamW fallback on 1D params (default (0.9, 0.95))
        adamw_eps:    epsilon for the AdamW fallback (default 1e-8)
    """

    def __init__(self, params, lr=2e-2, momentum=0.95, nesterov=True,
                 ns_steps=5, weight_decay=1e-2,
                 adamw_betas=(0.9, 0.95), adamw_eps=1e-8):
        defaults = dict(
            lr=lr, momentum=momentum, nesterov=nesterov,
            ns_steps=ns_steps, weight_decay=weight_decay,
            adamw_betas=adamw_betas, adamw_eps=adamw_eps,
        )
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group['lr']
            momentum = group['momentum']
            nesterov = group['nesterov']
            ns_steps = group['ns_steps']
            weight_decay = group['weight_decay']
            beta1, beta2 = group['adamw_betas']
            eps = group['adamw_eps']

            for p in group['params']:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]

                if p.ndim >= 2:
                    # ── Matrix parameter: Muon update ──
                    # One buffer per matrix (Adam would need two)
                    if len(state) == 0:
                        state['momentum_buffer'] = torch.zeros_like(p)
                    buf: torch.Tensor = state['momentum_buffer']

                    # Momentum buffer (Nesterov: look-ahead update)
                    buf.mul_(momentum).add_(grad)
                    update = grad.add(buf, alpha=momentum) if nesterov else buf

                    # Orthogonalize as a 2D matrix: a no-op reshape for a Linear
                    # weight, (out, in*kh*kw) for a conv filter
                    update = newton_schulz(update.reshape(len(update), -1), steps=ns_steps)

                    # Shape scale (Keller Jordan's reference): scale tall
                    # matrices (fan-out > fan-in) up, leave wide ones alone
                    rows, cols = update.shape
                    scale = max(1.0, rows / cols) ** 0.5

                    # Decoupled weight decay (same pattern as AdamW)
                    if weight_decay > 0:
                        p.mul_(1 - lr * weight_decay)

                    # Apply the orthogonalized update
                    p.add_(update.reshape(p.shape), alpha=-lr * scale)

                else:
                    # ── 1D parameter (bias, norm gain): AdamW fallback ──
                    if len(state) == 0:
                        state['step'] = 0
                        state['exp_avg'] = torch.zeros_like(p)
                        state['exp_avg_sq'] = torch.zeros_like(p)
                    state['step'] += 1
                    exp_avg: torch.Tensor = state['exp_avg']
                    exp_avg_sq: torch.Tensor = state['exp_avg_sq']

                    exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                    exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                    bias1 = 1 - beta1 ** state['step']
                    bias2 = 1 - beta2 ** state['step']
                    denom = (exp_avg_sq.sqrt() / math.sqrt(bias2)).add_(eps)
                    step_size = lr / bias1

                    if weight_decay > 0:
                        p.mul_(1 - lr * weight_decay)

                    p.addcdiv_(exp_avg, denom, value=-step_size)

        return loss

## Validation

`newton_schulz` is checked against an exact SVD (same singular vectors, singular values pulled into a band around 1), Muon by training a tiny linear model, and — on PyTorch ≥ 2.9 — the matrix update is compared with `torch.optim.Muon`.

Until your implementations are in place these tests will fail — that is expected.

In [ ]:
def test_newton_schulz():
    """Newton–Schulz keeps the singular vectors and pushes singular values to ~1."""
    print("Testing newton_schulz...", end=" ")

    torch.manual_seed(42)
    for m, n in ((32, 64), (64, 32)):
        X = torch.randn(m, n)
        O = newton_schulz(X, steps=5)

        # Express O in X's singular basis: D = Uᵀ O V should be diagonal
        U, _, Vh = torch.linalg.svd(X, full_matrices=False)
        D = U.T @ O @ Vh.T
        sv = D.diag()
        off = (D - torch.diag(sv)).abs().max().item()

        # off-diagonal is ~1e-6 in float32; ~0.02 if you iterate in bfloat16 like
        # Keller Jordan's GPU code — both are fine
        ok = O.shape == X.shape and off < 0.05 and sv.min() > 0.5 and sv.max() < 1.5
        if not ok:
            print(f"FAIL  ({m}x{n}: singular values in [{sv.min():.3f}, {sv.max():.3f}], "
                  f"off-diagonal {off:.1e})")
            return
    print(f"PASS  (singular values in [{sv.min():.3f}, {sv.max():.3f}], "
          f"exact orthogonalization would give 1)")

In [ ]:
def test_muon():
    """Muon should reduce the loss on a small linear model."""
    print("Testing Muon...", end=" ")

    torch.manual_seed(42)
    B, D_in, D_out = 32, 16, 8

    model = nn.Linear(D_in, D_out)
    opt = MyMuon(model.parameters(), lr=0.02, momentum=0.95, weight_decay=1e-3)

    X = torch.randn(B, D_in)
    y = torch.randn(B, D_out)

    initial_loss = ((model(X) - y) ** 2).mean().item()

    for _ in range(200):
        loss = ((model(X) - y) ** 2).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()

    final_loss = ((model(X) - y) ** 2).mean().item()

    if final_loss < initial_loss * 0.5:
        print(f"PASS  (loss: {initial_loss:.4f} -> {final_loss:.4f})")
    else:
        print(f"FAIL  (loss: {initial_loss:.4f} -> {final_loss:.4f})")


def test_muon_matches_torch():
    """Matrix updates should agree with torch.optim.Muon (PyTorch >= 2.9)."""
    print("Testing Muon vs torch.optim.Muon...", end=" ")
    if not hasattr(torch.optim, "Muon"):
        print("SKIPPED (needs PyTorch >= 2.9)")
        return

    torch.manual_seed(0)
    w0 = torch.randn(32, 16)                     # tall: exercises the shape scale
    grads = [torch.randn(32, 16) for _ in range(5)]
    kw = dict(lr=0.02, momentum=0.95, nesterov=True, weight_decay=0.01)

    results = []
    for opt_cls in (MyMuon, torch.optim.Muon):
        w = w0.clone().requires_grad_(True)
        opt = opt_cls([w], **kw)
        for g in grads:
            w.grad = g.clone()
            opt.step()
        results.append(w.detach() - w0)

    # torch runs Newton–Schulz in bfloat16, so expect ~1% agreement, not 1e-6
    rel = ((results[0] - results[1]).norm() / results[1].norm()).item()
    if rel < 0.05:
        print(f"PASS  (relative difference of the total update: {rel:.2%})")
    else:
        print(f"FAIL  (relative difference of the total update: {rel:.2%})")


test_newton_schulz()
test_muon()
test_muon_matches_torch()

In [ ]:
# Auto-grade (one-time setup from the repo root: pip install -e python/)
from torchleet import check
check("muon", newton_schulz=newton_schulz, MyMuon=MyMuon)